Task 6 — measured efficiency of the baseline pair-force calculation.

Required comparisons: pure Python double loop, NumPy vectorized, compiled C.
Each method uses the same fixed-seed positions, baseline constants (k=15, r0=0.8,
r_c=2.0), and the baseline's inclusive neighbour rule 0 < distance <= r_c.
Each timing has a warm-up followed by >=5 repeats; the median wall time is reported.
The C code is compiled at runtime with -O2 when gcc/clang/cc is available.

The public `pairwise_dist`, `pair_forces`, and `pair_forces_vectorized` APIs are kept
compatible with Baseline_Simulator_Sridatt.ipynb. `pair_forces` preserves the Task 1
matrix + unique-pair accumulation method; `forces_python` is the pure scalar double-loop
method benchmarked to represent the project rubric's Python double loop.

Outputs (in --output-dir):
    task6_timing_results.csv
    task6_scaling_exponents.csv
    task6_wall_time_vs_N.png
    task6_speedup_vs_N.png

In [ ]:
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import ctypes
import csv
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path
from typing import Callable, Dict, Iterable

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Baseline constants and names
k = 15.0
r0 = 0.8
r_c = 2.0
L = 10.0
DEFAULT_SIZES = [50, 200, 500, 1000]
SEED = 0
DEFAULT_REPEATS = 5


def pairwise_dist(pos: np.ndarray, r_c: float = r_c):
    """Baseline API: displacement, distance, and cutoff-neighbor matrices."""
    d = pos[:, None, :] - pos[None, :, :]
    dist = np.sum(np.power(d, 2), axis=-1) ** (1 / 2)
    neighbors = (dist > 0) & (dist <= r_c)
    return d, dist, neighbors


def pair_forces(pos: np.ndarray, k: float = k, r0: float = r0,
                r_c: float = r_c) -> np.ndarray:
    """Baseline Task 1 implementation, retained for interoperability."""
    F = np.zeros_like(pos)
    d, dist, neighbors = pairwise_dist(pos, r_c)
    scalar = pos.dtype.type
    k_t, r0_t = scalar(k), scalar(r0)
    for i in range(pos.shape[0]):
        for j in range(i + 1, pos.shape[0]):
            if neighbors[i, j]:
                f_vec = -k_t * (dist[i, j] - r0_t) * d[i, j] / dist[i, j]
                F[i] += f_vec
                F[j] -= f_vec
    return F


def forces_python(pos: np.ndarray, k: float = k, r0: float = r0,
                  r_c: float = r_c) -> np.ndarray:
    """Pure Python scalar double loop; visits each unordered pair once."""
    points = np.asarray(pos, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("pos must have shape (N, 2)")
    n = len(points)
    F = np.zeros((n, 2), dtype=np.float64)
    for i in range(n):
        xi, yi = points[i, 0], points[i, 1]
        for j in range(i + 1, n):
            dx = xi - points[j, 0]
            dy = yi - points[j, 1]
            dist = math.sqrt(dx * dx + dy * dy)
            if 0.0 < dist <= r_c:  # same inclusive cutoff as baseline Task 1
                magnitude = k * (dist - r0)
                fx = -magnitude * dx / dist
                fy = -magnitude * dy / dist
                F[i, 0] += fx
                F[i, 1] += fy
                F[j, 0] -= fx
                F[j, 1] -= fy
    return F


def pair_forces_vectorized(pos: np.ndarray, k: float = k, r0: float = r0,
                           r_c: float = r_c) -> np.ndarray:
    """Vectorized equivalent of the force formula used in baseline."""
    p = np.asarray(pos)
    d, dist, neighbors = pairwise_dist(p, r_c)
    tiny = p.dtype.type(1e-12)
    u = d / (dist[:, :, None] + tiny)
    fmag = -p.dtype.type(k) * (dist - p.dtype.type(r0)) * neighbors
    F = np.sum(fmag[:, :, None] * u, axis=1)
    return F


_C_SOURCE = r'''#include <math.h>

/* Baseline-compatible 2-D spring pair forces; cutoff is inclusive. */
void pair_forces_c(const double *pos, double *F, int N,
                   double k, double r0, double rc) {
    for (int q = 0; q < 2 * N; ++q) F[q] = 0.0;
    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {
            const double dx = pos[2*i] - pos[2*j];
            const double dy = pos[2*i + 1] - pos[2*j + 1];
            const double r = sqrt(dx*dx + dy*dy);
            if (r > 0.0 && r <= rc) {
                const double mag = k * (r - r0);
                const double fx = -mag * dx / r;
                const double fy = -mag * dy / r;
                F[2*i]     += fx;
                F[2*i + 1] += fy;
                F[2*j]     -= fx;
                F[2*j + 1] -= fy;
            }
        }
    }
}
'''


def compile_c_library(output_dir: Path):
    """Compile a shared C library. Returns (callable force function or None, status)."""
    compiler = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
    if compiler is None:
        return None, "No gcc/clang/cc compiler found; compiled-C timing is unavailable on this machine."

    source_path = output_dir / "task6_pairforce.c"
    source_path.write_text(_C_SOURCE, encoding="utf-8")
    if sys.platform == "win32":
        lib_path = output_dir / "task6_pairforce.dll"
        command = [compiler, "-O2", "-shared", "-o", str(lib_path), str(source_path), "-lm"]
    elif sys.platform == "darwin":
        lib_path = output_dir / "libtask6_pairforce.dylib"
        command = [compiler, "-O2", "-dynamiclib", "-fPIC", "-o", str(lib_path), str(source_path), "-lm"]
    else:
        lib_path = output_dir / "libtask6_pairforce.so"
        command = [compiler, "-O2", "-shared", "-fPIC", "-o", str(lib_path), str(source_path), "-lm"]

    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0 or not lib_path.exists():
        detail = completed.stderr.strip() or completed.stdout.strip() or "unknown compiler error"
        return None, f"C compilation failed; compiled-C timing is unavailable. Details: {detail}"

    lib = ctypes.CDLL(str(lib_path))
    lib.pair_forces_c.argtypes = [
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        np.ctypeslib.ndpointer(dtype=np.float64, flags="C_CONTIGUOUS"),
        ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_double,
    ]
    lib.pair_forces_c.restype = None

    def forces_c(pos: np.ndarray, k: float = k, r0: float = r0,
                 r_c: float = r_c) -> np.ndarray:
        p = np.ascontiguousarray(pos, dtype=np.float64)
        F = np.zeros_like(p)
        lib.pair_forces_c(p, F, len(p), float(k), float(r0), float(r_c))
        return F

    return forces_c, f"Compiled C successfully with {Path(compiler).name} -O2."


def median_time(function: Callable[[np.ndarray], np.ndarray], pos: np.ndarray,
                repeats: int = DEFAULT_REPEATS) -> tuple[float, list[float]]:
    """Warm up once, then return median and all repeated wall-clock times."""
    function(pos)
    measurements = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(pos)
        measurements.append(time.perf_counter() - start)
    return float(np.median(measurements)), measurements


def write_csv(path: Path, rows: Iterable[dict], fieldnames: list[str] | None = None) -> None:
    rows = list(rows)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def run_task6(sizes: list[int] | None = None, repeats: int = DEFAULT_REPEATS,
              random_seed: int = SEED, output_dir: str | Path = "task6_results") -> dict:
    if sizes is None:
        sizes = DEFAULT_SIZES.copy()
    if any(n < 2 for n in sizes):
        raise ValueError("Every particle count must be >=2")
    if repeats < 5:
        warnings.warn("The rubric requests at least five timings; use repeats >= 5 for final results.")
    outdir = Path(output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    c_func, c_message = compile_c_library(outdir)
    print(c_message)

    methods: Dict[str, Callable[[np.ndarray], np.ndarray]] = {
        "Python double loop": forces_python,
        "NumPy vectorized": pair_forces_vectorized,
    }
    if c_func is not None:
        methods["Compiled C (-O2)"] = c_func

    timing_rows = []
    per_size = {}
    for n in sizes:
        # One fixed position array per N is reused by every implementation.
        rng = np.random.default_rng(random_seed + n)
        pos = rng.random((n, 2), dtype=np.float64) * L

        # Baseline method gives a direct compatibility cross-check; the Python scalar
        # loop is the benchmark reference requested in the rubric.
        ref = forces_python(pos)
        baseline_check = pair_forces(pos)
        if not np.allclose(ref, baseline_check, rtol=1e-8, atol=1e-8):
            raise AssertionError(f"Baseline pair_forces disagrees with scalar loop at N={n}")

        measured = {}
        for name, func in methods.items():
            out = func(pos)
            max_diff = float(np.max(np.abs(out - ref))) if out.size else 0.0
            if not np.allclose(out, ref, rtol=1e-8, atol=1e-8):
                raise AssertionError(f"{name} disagrees with Python reference at N={n}; max abs diff={max_diff:.3e}")
            median_s, samples = median_time(func, pos, repeats=repeats)
            measured[name] = {"median_s": median_s, "samples": samples, "max_abs_diff": max_diff,
                              "net_force_norm": float(np.linalg.norm(out.sum(axis=0)))}

        reference_time = measured["Python double loop"]["median_s"]
        per_size[n] = measured
        for name, values in measured.items():
            timing_rows.append({
                "N": n,
                "implementation": name,
                "repeats": repeats,
                "sample_times_seconds": ";".join(f"{t:.9g}" for t in values["samples"]),
                "median_wall_time_seconds": values["median_s"],
                "minimum_wall_time_seconds": min(values["samples"]),
                "maximum_wall_time_seconds": max(values["samples"]),
                "speedup_vs_python_double_loop": reference_time / values["median_s"],
                "max_abs_force_difference_vs_python": values["max_abs_diff"],
                "norm_of_total_internal_force": values["net_force_norm"],
            })
        print(f"N={n}: " + ", ".join(
            f"{name} {v['median_s']:.6g}s ({reference_time / v['median_s']:.2f}x)"
            for name, v in measured.items()
        ))

    write_csv(outdir / "task6_timing_results.csv", timing_rows)

    # Empirical log-log exponent: t ~ N^p, estimated from the median measured points.
    exponent_rows = []
    for name in methods:
        times = np.array([per_size[n][name]["median_s"] for n in sizes], dtype=float)
        if len(sizes) >= 2 and np.all(times > 0):
            exponent = float(np.polyfit(np.log(np.asarray(sizes, dtype=float)), np.log(times), 1)[0])
        else:
            exponent = float("nan")
        exponent_rows.append({
            "implementation": name,
            "measured_loglog_scaling_exponent": exponent,
            "interpretation": "approximately quadratic pair search if exponent is near 2; small-N timing noise can shift this estimate",
        })
    write_csv(outdir / "task6_scaling_exponents.csv", exponent_rows)

    if plt is not None:
        fig, ax = plt.subplots(figsize=(8.5, 5.5))
        for name in methods:
            ax.plot(sizes, [per_size[n][name]["median_s"] for n in sizes], marker="o", label=name)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Number of particles, N")
        ax.set_ylabel("Median wall-clock time (seconds)")
        ax.set_title("Pair-force wall time versus system size")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(outdir / "task6_wall_time_vs_N.png", dpi=180)
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8.5, 5.5))
        for name in methods:
            speedups = [per_size[n]["Python double loop"]["median_s"] / per_size[n][name]["median_s"] for n in sizes]
            ax.plot(sizes, speedups, marker="o", label=name)
        ax.axhline(1.0, linestyle="--", linewidth=1, label="Python reference (1×)")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Number of particles, N")
        ax.set_ylabel("Speedup = Python median / method median")
        ax.set_title("Pair-force speedup versus Python double loop")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(outdir / "task6_speedup_vs_N.png", dpi=180)
        plt.close(fig)
    else:
        print("matplotlib is not installed; CSV timing tables were written, but plots were skipped.")

    largest_n = max(sizes)
    slowest_ref = per_size[largest_n]["Python double loop"]["median_s"]
    fastest_name, fastest_data = min(per_size[largest_n].items(), key=lambda item: item[1]["median_s"])
    print("\nTASK 6 — RESULTS SUMMARY")
    print(f"Same positions used within each N; cutoff uses baseline condition 0 < r <= {r_c}.")
    print(f"At N={largest_n}, fastest measured method was {fastest_name} at {fastest_data['median_s']:.6g} s,")
    print(f"which is {slowest_ref / fastest_data['median_s']:.2f}x faster than the Python double-loop median on this run.")
    print("All reported speedups are measured; no pre-assumed ranking is used.")
    print("Computational bottleneck: the pair search considers N(N-1)/2 unordered pairs, so the work is O(N^2).")
    print("The vectorized method removes per-pair Python overhead but creates O(N^2) temporary arrays; C avoids those arrays but still performs O(N^2) pair checks.")
    print("Measured log-log scaling exponents:")
    for row in exponent_rows:
        print(f"  {row['implementation']}: p={row['measured_loglog_scaling_exponent']:.3f}")

    report_lines = [
        "TASK 6 — COMPUTATIONAL EFFICIENCY: RESULTS AND JUSTIFICATION",
        "============================================================",
        f"Measured on this runtime with N values {', '.join(map(str, sizes))}; repeats per method={repeats}; fixed seed base={random_seed}.",
        f"Baseline force constants: k={k}, r0={r0}, cutoff r_c={r_c}, inclusive neighbor rule 0 < r <= r_c.",
        "The same positions are reused across implementations for each N. Each method is warmed up once; reported times are medians of measured repeats.",
        "",
        "MEASURED TIMING / SPEEDUP TABLE",
        "N | Implementation | Median seconds | Speedup vs Python | Max abs force difference | Net-force norm",
        "--|--|--:|--:|--:|--:",
    ]
    for row in timing_rows:
        report_lines.append(
            f"{row['N']} | {row['implementation']} | {row['median_wall_time_seconds']:.6e} | "
            f"{row['speedup_vs_python_double_loop']:.3f}x | {row['max_abs_force_difference_vs_python']:.3e} | "
            f"{row['norm_of_total_internal_force']:.3e}"
        )
    report_lines += ["", "EMPIRICAL SCALING", "Implementation | Log-log exponent p in t approximately proportional to N^p", "--|--:"]
    for row in exponent_rows:
        report_lines.append(f"{row['implementation']} | {row['measured_loglog_scaling_exponent']:.3f}")
    report_lines += [
        "",
        "INTERPRETATION / JUSTIFICATION",
        f"- At N={largest_n}, {fastest_name} was the fastest measured implementation ({fastest_data['median_s']:.6e} s), "
        f"giving {slowest_ref / fastest_data['median_s']:.3f}x speedup relative to the Python double loop in this run.",
        "- The pair calculation visits N(N-1)/2 unordered pairs, so the direct pair search has O(N^2) work. The measured log-log exponent is empirical and can deviate from 2 due to cache behavior, vectorization, compiler optimization, and timing noise.",
        "- NumPy reduces Python-loop overhead but allocates O(N^2) pairwise arrays; compiled C avoids those large vectorized temporaries while retaining O(N^2) pair checks. The fastest method is decided from measurements, not assumed in advance.",
        "- Force-difference and net-force columns are correctness checks: a faster result should not be credited if it fails to agree with the common Python reference within the script's tolerance.",
        "- Small-N results are especially sensitive to overhead. Use the full timing-vs-N and speedup-vs-N plots, not only the largest-N result, to justify the implementation choice.",
        "",
        "REPORT LIMITATION",
        "Wall times depend on the current Colab runtime type/load and may vary between sessions. Include these measured values in the report and identify the runtime; rerun the benchmark if the runtime changes.",
        "",
        "OUTPUT FILES",
        "- task6_timing_results.csv: raw repeat samples, medians, speedups, and force checks.",
        "- task6_scaling_exponents.csv: measured log-log scaling exponents.",
        "- task6_wall_time_vs_N.png and task6_speedup_vs_N.png: plots.",
        "- task6_pairforce.c and compiled library (when a C compiler is available): C benchmark source and build output.",
    ]
    report_path = outdir / "task6_interpretation.txt"
    report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
    print(f"Saved timing tables, plots, and interpretation: {outdir.resolve()}")
    return {"per_size": per_size, "timing_rows": timing_rows, "exponent_rows": exponent_rows,
            "output_dir": outdir, "methods": list(methods)}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--sizes", type=int, nargs="+", default=DEFAULT_SIZES,
                        help="particle counts; rubric defaults are 50 200 500 1000")
    parser.add_argument("--repeats", type=int, default=DEFAULT_REPEATS,
                        help="timed repeats per method and N (rubric requires at least 5)")
    parser.add_argument("--seed", type=int, default=SEED, help="fixed seed for reproducibility")
    parser.add_argument("--output-dir", default="task6_results", help="directory for data and plots")
    args = parser.parse_args()
    run_task6(args.sizes, args.repeats, args.seed, args.output_dir)


if __name__ == "__main__":
    main()